In [10]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "lca-lc-foundation")

print(f"LangSmith tracing: {os.environ['LANGSMITH_TRACING']}")
print(f"LangSmith project: {os.environ['LANGSMITH_PROJECT']}")

LangSmith tracing: true
LangSmith project: lca-lc-foundation


In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [3]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [5]:
from langchain.messages import HumanMessage

config = {
    "configurable": {"thread_id": "1"},
    "run_name": "personal-chef",
    "tags": ["module-1", "personal-chef"],
    "metadata": {"application": "personal-chef", "module": "module-1"},
}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print(response['messages'][-1].content)

Here are three tasty ways to turn your leftover chicken and rice into a fresh, satisfying meal.  
Feel free to pick the one that matches your mood (quick & skillet‑friendly, comforting baked, or a one‑pot stovetop classic).  If you have any extra ingredients (veggies, herbs, sauces, cheese, etc.) let me know and I can tweak the recipes for you.

---

## 1. Classic Chicken Fried Rice  
**Prep time:** 5 min | **Cook time:** 10 min | **Serves:** 4‑5

| Ingredient | Amount (adjust to taste) |
|------------|--------------------------|
| Leftover cooked chicken, shredded or diced | 1½ cups |
| Leftover cooked rice (cold, preferably day‑old) | 3 cups |
| Olive oil | 1 tsp |
| Sesame oil | 1 tsp |
| Frozen peas & carrots (or any quick‑cook veg) | 1 cup |
| Green onions, sliced | 4 – 5 sticks |
| Garlic, minced | 2 cloves |
| Eggs, beaten | 2 |
| Soy sauce | 3 Tbsp |
| Salt & pepper | to taste |
| Optional: chili flakes or sriracha | pinch |

**Instructions**

1. **Heat the pan**: In a large wo

In [6]:
follow_up = agent.invoke(
    {"messages": [HumanMessage(content="Can you give me the instructions for the second option, and suggest a vegetable side?")]},
    config,
)

print(follow_up["messages"][-1].content)

### One‑Pot Chicken & Rice Skillet  
*(Serves 4, about 20 min total)*

| Ingredient | Amount |
|------------|--------|
| Leftover chicken, diced | 2 cups |
| Leftover rice | 2 cups |
| Chicken broth or water | 2 cups |
| Onion, diced | ½ cup |
| Bell pepper, diced | ½ cup |
| Frozen peas | ½ cup |
| Garlic powder (or 1 clove minced) | 1 tsp |
| Paprika | ½ tsp |
| Salt & pepper | to taste |
| Olive oil | 1 Tbsp |

#### Step‑by‑Step Instructions

1. **Prepare the pan**  
   - Heat a large skillet or sauté pan over medium heat.  
   - Add the tablespoon of olive oil and swirl to coat the bottom.

2. **Sauté aromatics**  
   - Add the diced onion (and bell pepper if you like).  
   - Cook, stirring occasionally, until the onion is translucent and the pepper is softened, about **3 minutes**.

3. **Season**  
   - Sprinkle in the garlic powder, paprika, salt, and pepper.  
   - Stir for **30 seconds** to let the spices bloom.

4. **Add rice and liquid**  
   - Stir in the leftover rice, bre

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken and rice. What can I make?', additional_kwargs={}, response_metadata={}, id='ce5d482c-f9df-4fbc-b761-908ce04220c7'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "We need to search web for recipes with chicken and rice. Provide suggestions. Then ask if wants instructions. We'll do web_search.", 'tool_calls': [{'id': 'fc_0b82236d-0d35-43f7-ba9c-0c1de7fe6004', 'function': {'arguments': '{"query":"chicken rice recipe leftover chicken and rice"}', 'name': 'web_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 194, 'total_tokens': 251, 'completion_time': 0.057344929, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.01082378, 'prompt_tokens_details': None, 'queue_time': 0.048354657, 'total_time': 0.068168709}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e189667b30', 'service_tier': 'on_demand', 'fi

## Change the domain

The same pattern can search for a learning project instead of a recipe. Replace this prompt with a topic you care about, then keep a thread id to ask follow-up questions.

In [8]:
learning_prompt = """
You are a practical learning-project scout. The user will give you a topic and the time they have.
Use web search to find current, trustworthy tutorials or project ideas that fit.
Return a short ranked list with links and explain which option best fits the user's constraints.
Remember the user's topic and constraints so you can answer follow-up questions in the same thread.
"""

learning_agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[web_search],
    system_prompt=learning_prompt,
    checkpointer=InMemorySaver(),
)

learning_config = {"configurable": {"thread_id": "learning-project-1"}}
learning_response = learning_agent.invoke(
    {"messages": [HumanMessage(content="I want to learn practical data visualization this weekend. What project should I build?")]},
    learning_config,
)

print(learning_response["messages"][-1].content)

**Practical data‑visualization projects you can finish in a weekend**

| Rank | Project | What you’ll build | Why it’s a good weekend pick | Link |
|------|---------|------------------|------------------------------|------|
| **1** | **Airline analysis with Python & Seaborn** | Interactive line & bar charts showing flight price trends, on‑time performance, and seat‑class occupancy. | • Python is widely taught and the tutorial gives step‑by‑step code.<br>• The dataset (open‑source flight data) is small (≈ 50 k rows) and loads in < 1 min.<br>• You’ll learn data‑cleaning, grouping, and visualizing in < 4 hrs. | <https://www.codecademy.com/projects/subject/data-visualization#airline-analysis> |
| **2** | **Excel flight‑pricing dashboard** | A dashboard with slicers, pivot tables, and charts that shows how ticket prices vary by day, month, and destination. | • No coding required – just Excel formulas and the built‑in chart tools.<br>• The tutorial walks you through importing the dataset, cl

In [9]:
learning_follow_up = learning_agent.invoke(
    {"messages": [HumanMessage(content="Which option did you rank first, and what should I do first tonight?")]},
    learning_config,
)

print(learning_follow_up["messages"][-1].content)

**You’re looking at the top‑ranked project:**

> **Airline analysis with Python & Seaborn**  
> (interactive line & bar charts showing flight price trends, on‑time performance, and seat‑class occupancy)

---

## First thing to do tonight

1. **Set up a quick Python environment**  
   - If you already have Anaconda, open *Anaconda Prompt* or your terminal.  
   - If you don’t, install Miniconda (or just use a lightweight `venv`):
     ```bash
     # Create and activate a new env
     python -m venv vizenv
     source vizenv/bin/activate   # (Windows: vizenv\Scripts\activate)
     # Install required packages
     pip install pandas seaborn matplotlib
     ```

2. **Download the dataset**  
   - The tutorial uses the “OpenFlights” dataset (CSV). Grab it from the project page or directly:
     ```bash
     curl -O https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat
     # Or use the link provided in the tutorial to a CSV of flight prices
     ```
   - Save it in

In [11]:
def chat(message: str) -> str:
    result = agent.invoke(
        {"messages": [HumanMessage(content=message)]},
        config,
    )
    answer = result["messages"][-1].content
    print(answer)
    return answer


chat("What is one quick dinner I can make with the ingredients I have?")

**Quick 15‑minute Chicken Fried Rice**  
*(Makes 4–5 servings)*  

### Ingredients  
| Item | Amount |
|------|--------|
| Leftover cooked chicken, shredded or diced | 1 ½ cups |
| Leftover cooked rice (cold, preferably day‑old) | 3 cups |
| Olive oil | 1 tsp |
| Sesame oil (optional) | 1 tsp |
| Frozen peas & carrots (or any quick‑cook veg) | 1 cup |
| Green onions, sliced | 4 – 5 sticks |
| Garlic, minced | 2 cloves |
| Eggs, beaten | 2 |
| Soy sauce | 3 Tbsp |
| Salt & pepper | to taste |
| Chili flakes or sriracha (optional) | pinch |

### Instructions  

1. **Heat the pan**  
   - Warm the olive oil in a large wok or skillet over medium‑high heat.

2. **Sauté aromatics**  
   - Add minced garlic and 1 Tbsp of sliced green onions.  
   - Cook 30 s until fragrant.

3. **Add the veggies**  
   - Stir‑fry the frozen peas & carrots for 2 min until just thawed.

4. **Scramble the eggs**  
   - Push the mixture to one side of the pan, pour the beaten eggs into the empty space, and scramb

'**Quick 15‑minute Chicken Fried Rice**  \n*(Makes 4–5 servings)*  \n\n### Ingredients  \n| Item | Amount |\n|------|--------|\n| Leftover cooked chicken, shredded or diced | 1\u202f½\u202fcups |\n| Leftover cooked rice (cold, preferably day‑old) | 3\u202fcups |\n| Olive oil | 1\u202ftsp |\n| Sesame oil (optional) | 1\u202ftsp |\n| Frozen peas & carrots (or any quick‑cook veg) | 1\u202fcup |\n| Green onions, sliced | 4\u202f–\u202f5\u202fsticks |\n| Garlic, minced | 2\u202fcloves |\n| Eggs, beaten | 2 |\n| Soy sauce | 3\u202fTbsp |\n| Salt & pepper | to taste |\n| Chili flakes or sriracha (optional) | pinch |\n\n### Instructions  \n\n1. **Heat the pan**  \n   - Warm the olive oil in a large wok or skillet over medium‑high heat.\n\n2. **Sauté aromatics**  \n   - Add minced garlic and 1\u202fTbsp of sliced green onions.  \n   - Cook 30\u202fs until fragrant.\n\n3. **Add the veggies**  \n   - Stir‑fry the frozen peas & carrots for 2\u202fmin until just thawed.\n\n4. **Scramble the eggs** 